In [0]:
dbutils.widgets.dropdown(name = "environment", defaultValue= "dev",choices= ["dev","prod","qa"],label = "select Environment")
env = dbutils.widgets.get("environment")
# print(env)

In [0]:
silverTablName = f"saleslake_{env}.silver_{env}.cleancustomer"
print(silverTablName)
brtable_name = f"saleslake_{env}.bronze_{env}.rawCustomer"
print(brtable_name)
# srcFileLoc=f"s3://saleslakekir/saleslake/{env}/src_file/"
# print(srcFileLoc)

In [0]:
spark.sql(f"""INSERT INTO {silverTablName}
SELECT DISTINCT   
    CAST(TRIM(customer_id) AS INTEGER) as customer_id ,
    UPPER(TRIM(customer_name)) as customer_name,
    UPPER(TRIM(email)) as email,
    UPPER(TRIM(phone)) as phone,
    UPPER(TRIM(address)) as address,
    UPPER(TRIM(city)) as city,
    UPPER(TRIM(state)) as state,
    UPPER(TRIM(country)) as country,
    UPPER(TRIM(zip_code)) as zip_code,
    UPPER(TRIM(segment)) as segment,
    CURRENT_TIMESTAMP() as ingest_ts
FROM {brtable_name}
WHERE ingest_ts > (
                    SELECT coalesce(MAX(ingest_ts),TO_DATE('1990-01-01','yyyy-MM-dd')) 
                    FROM {silverTablName}
                    )
ORDER BY CAST(TRIM(customer_id) AS INTEGER)
""")


In [0]:
%sql
-- spark.sql(f"""INSERT INTO {silverTablName}
SELECT DISTINCT   
    CAST(TRIM(customer_id) AS INTEGER) as customer_id ,
    UPPER(TRIM(customer_name)) as customer_name,
    UPPER(TRIM(email)) as email,
    UPPER(TRIM(phone)) as phone,
    UPPER(TRIM(address)) as address,
    UPPER(TRIM(city)) as city,
    UPPER(TRIM(state)) as state,
    UPPER(TRIM(country)) as country,
    UPPER(TRIM(zip_code)) as zip_code,
    UPPER(TRIM(segment)) as segment,
    CURRENT_TIMESTAMP() as ingest_ts
FROM saleslake_qa.bronze_qa.rawcustomer
WHERE ingest_ts > (
                    SELECT coalesce(MAX(ingest_ts),TO_DATE('1990-01-01','yyyy-MM-dd')) 
                    FROM saleslake_qa.silver_qa.cleancustomer
                    )
ORDER BY CAST(TRIM(customer_id) AS INTEGER)
-- """)


In [0]:
%sql
-- # SELECT * FROM intellibi_catlog.intellibi_bronzE.fMonthlySales;
-- # %sql
-- # SELECT * FROM intellibi_catlog.intellibi_silver.fMonthlySales;
-- # %sql
-- # SELECT * FROM intellibi_catlog.intellibi_SILVER.fMonthlySales;
select * from saleslake_qa.silver_qa.cleancustomer;
select * from saleslake_dev.silver_dev.cleancustomer;
select * from saleslake_prod.silver_prod.cleancustomer